In [3]:
from access_parser import AccessParser
import pandas as pd
import re
from pathlib import Path

In [4]:
DATA_DIR = Path('/Users/ben/Projects/ADEPT/Data')

In [5]:
db = AccessParser("/Users/ben/Downloads/ANGIOontologyTraits_230926.accdb")

In [6]:
table = db.parse_table("angiosperm_traits")

In [7]:
df = pd.DataFrame(table)

In [8]:
df.shape

(1742, 27)

In [9]:
# Normalise column names first
df = df.rename(columns=lambda c: re.sub(
    r'\s+',
    '',
    str(c).strip().lower()
))

# columns like character1, character2...
character_cols = {
    int(re.search(r'\d+', c).group()): c
    for c in df.columns
    if re.fullmatch(r'character\d+', c)
}

# columns like trait1, trait2...
trait_cols = {
    int(re.search(r'\d+', c).group()): c
    for c in df.columns
    if re.fullmatch(r'trait\d+', c)
}

# Columns that aren't character/trait columns
id_cols = [
    c for c in df.columns
    if c not in character_cols.values()
    and c not in trait_cols.values()
]

rows = []

for i in sorted(set(character_cols) | set(trait_cols)):
    tmp = df[id_cols].copy()

    tmp["character"] = (
        df[character_cols[i]]
        if i in character_cols
        else pd.NA
    )

    tmp["trait"] = (
        df[trait_cols[i]]
        if i in trait_cols
        else pd.NA
    )

    rows.append(tmp)

long_df = pd.concat(rows, ignore_index=True)

# Remove empty character/trait combinations
long_df = long_df.dropna(
    subset=["character", "trait"],
    how="all"
)

In [10]:
long_df[["term", "category", "character", "trait"]]

,term,category,character,trait
0,v_form,arrangement,conduplicate,Leaf architecture
1,ciliate,architecture,ciliate,Indumentum
2,crenate,shape,crenate,Leaf margin
3,dentate,shape,dentate,Leaf margin
4,digitate,architecture,digitate,Leaf Arrangement
...,...,...,...,...
15331,white_flushed,coloration,white,Petal colour
15332,white_flecked,coloration,white,Petal colour
15347,semipellucid,Indumentum,pellucid,None
15357,substramineous,coloration,yellow,Petal colour


In [74]:
new = long_df[['term', 'category', 'character', 'trait']]

In [11]:
adept_df = pd.read_parquet('/Users/ben/Projects/ADEPT/Code/PrePub/ADEPT/adept/data/assets/traits.parquet')

In [12]:
adept_df[adept_df.term == 'perennial']

,term,category,character,trait,part,type,group
585,perennial,duration,perennial,life cycle,None,discrete,angiosperm


In [14]:
adept_df = adept_df[adept_df.group == 'angiosperm']

In [15]:
adept_df[adept_df.trait == 'habit']

,term,category,character,trait,part,type,group
13,acaulescent,architecture,tussock,habit,None,discrete,angiosperm
14,acaulous,architecture,tussock,habit,None,discrete,angiosperm
20,aerial-rooted,architecture,climber/scrambler,habit,None,discrete,angiosperm
41,arboreous,growth_form,tree,habit,None,discrete,angiosperm
65,bush,growth_form,shrub,habit,None,discrete,angiosperm
...,...,...,...,...,...,...,...
2532,stipe,structure,tree,habit,None,discrete,angiosperm
2593,submerged,location,submerged,habit,None,discrete,angiosperm
2657,rhizome,structure,erect leafy/tussock,habit,None,discrete,angiosperm
2752,woody-based,growth_form,suffrutex,habit,None,discrete,angiosperm


In [16]:
new = long_df[['term', 'category', 'character', 'trait']]

In [17]:
new = new.map(lambda x: x.lower() if isinstance(x, str) else x)

In [18]:
new[~new.trait.isin(adept_df.trait)].trait.value_counts()

typos = {
    'pernnial organ': 'perennial organ',
    'life form': 'life cycle',
    'leav shape': 'leaf shape',
    'leas base': 'leaf base',
    'leas apex': 'leaf apex',
    'polination': 'pollination'
}

new["trait"] = new["trait"].replace(typos)


In [19]:
lookup = (
    adept_df[["trait", "part", "type"]]
    .drop_duplicates()
    .drop_duplicates("trait")
)

new = new.merge(
    lookup,
    on="trait",
    how="left"
)

In [20]:
# Update any traits that have colour - setting part and type=colour
mask = (
    new["trait"].str.contains("colour", case=False, na=False)
    & new["part"].isna()
    & new["type"].isna()
)

new.loc[mask, "part"] = (
    new.loc[mask, "trait"]
    .str.replace("colour", "", case=False, regex=False)
    .str.strip()
)

new.loc[mask, "type"] = "colour"

In [21]:
new[new.term.str.contains('_')]

,term,category,character,trait,part,type
0,v_form,arrangement,conduplicate,leaf architecture,leaf,discrete
20,aerial_rooted,architecture,climber/scrambler,habit,None,discrete
46,awl_shaped,shape,aciculate,leaf shape,leaf,discrete
52,berry_like,architecture,berry,fruit type,fruit,discrete
87,cordate_ovate,shape,ovate,leaf shape,leaf,discrete
...,...,...,...,...,...,...
3879,black_girdled,coloration,black,petal colour,petal,colour
3880,purple_flecked,coloration,purple,petal colour,petal,colour
3881,violet_flecked,coloration,purple,petal colour,petal,colour
3882,white_flushed,coloration,white,petal colour,petal,colour


In [22]:
new["term"] = new["term"].str.replace("_", "-", regex=False)

In [23]:
new['group'] = "angiosperm"

In [24]:
# Filter out any where the character is na
new = new[~new["character"].isna()]

In [25]:
new[new["term"].isna()]

,term,category,character,trait,part,type,group


In [26]:
# Remove and dupes where the different is just an 's' on the end
s = new["term"]

mask = s.str.endswith("s") & s.str[:-1].isin(s)

new = new.loc[~mask]

In [27]:
values = set(new["character"].dropna())

mask = (
    new["character"].str.endswith("s", na=False)
    & new["character"].str[:-1].isin(values)
)

new.loc[mask, "character"] = new.loc[mask, "character"].str[:-1]

In [28]:
new

,term,category,character,trait,part,type,group
0,v-form,arrangement,conduplicate,leaf architecture,leaf,discrete,angiosperm
1,ciliate,architecture,ciliate,indumentum,leaf,discrete,angiosperm
2,crenate,shape,crenate,leaf margin,leaf,discrete,angiosperm
3,dentate,shape,dentate,leaf margin,leaf,discrete,angiosperm
4,digitate,architecture,digitate,leaf arrangement,leaf,discrete,angiosperm
...,...,...,...,...,...,...,...
3882,white-flushed,coloration,white,petal colour,petal,colour,angiosperm
3883,white-flecked,coloration,white,petal colour,petal,colour,angiosperm
3884,semipellucid,indumentum,pellucid,None,NaN,NaN,angiosperm
3885,substramineous,coloration,yellow,petal colour,petal,colour,angiosperm


In [29]:
# The ones with type set are the ones we want

new[~new.type.isna()].to_parquet(DATA_DIR / 'traits_260923.parquet', index=False)

In [31]:
# df.loc[df['character'] == 'ectzoochory', 'character'] = 'ectozoochory'

new[new.character == 'ectozoochory']

,term,category,character,trait,part,type,group
47,awned,shape,ectozoochory,dispersal mode,None,discrete,angiosperm
500,winglike,architecture,ectozoochory,dispersal mode,None,discrete,angiosperm
515,awnlike,shape,ectozoochory,dispersal mode,None,discrete,angiosperm
526,awn,structure,ectozoochory,dispersal mode,None,discrete,angiosperm
715,awn-point,structure,ectozoochory,dispersal mode,None,discrete,angiosperm
1296,awned,architecture,ectozoochory,dispersal mode,None,discrete,angiosperm
1350,awn-tipped,architecture,ectozoochory,dispersal mode,None,discrete,angiosperm
1869,bristle,structure,ectozoochory,dispersal mode,None,discrete,angiosperm
2096,bristly,pubescence,ectozoochory,dispersal mode,None,discrete,angiosperm
2294,bristled,shape,ectozoochory,dispersal mode,None,discrete,angiosperm


In [173]:
s = new["character"]

mask = s.str.endswith("s") & s.str[:-1].isin(s)

new.loc[mask]

,term,category,character,trait,part,type,group
